# Module 9: Thuật toán giải NLP: hàm phạt, hàm chắn, SQP, điểm trong — code

Notebook đi kèm site *Tối ưu hóa phi tuyến & lồi*. Chạy tuần tự từng ô (Colab có sẵn numpy, scipy, sympy, cvxpy). Tự giải tay trước rồi đối chiếu kết quả.

## Hàm phạt: bài tập 2, nghiệm và điều kiện số (dạng C)

min x²+y² với x+y=1: x=y=r/(2(1+r)); κ = 1+r.

In [1]:
import numpy as np
for r in (1, 10, 100, 1000):
    A = 2*np.eye(2) + r*np.ones((2, 2)); x = np.linalg.solve(A, r*np.ones(2))
    print(r, x.round(4), "vi pham", round(x.sum()-1, 5), "kappa", round(np.linalg.cond(A), 1))

1 [0.25 0.25] vi pham -0.5 kappa 2.0
10 [0.4545 0.4545] vi pham -0.09091 kappa 11.0
100 [0.495 0.495] vi pham -0.0099 kappa 101.0
1000 [0.4995 0.4995] vi pham -0.001 kappa 1001.0


## SQP đẳng thức: Bài tập 3 bằng hệ KKT (dạng E, F)

Bước đầu cho (1,1) chưa khả thi; sau đó hội tụ bậc hai về (0.7071, 0.7071), μ = √2 − 1.

In [2]:
import numpy as np
x = np.array([1., 0.]); lam = 0.0
for k in range(6):
    h = x@x - 1; g = 2*(x - 1); J = 2*x
    B = 2*np.eye(2) + 2*lam*np.eye(2) if k > 0 else 2*np.eye(2)
    K = np.block([[B, J.reshape(2, 1)], [J.reshape(1, 2), np.zeros((1, 1))]])
    sol = np.linalg.solve(K, -np.concatenate([g, [h]])); d, lam = sol[:2], sol[2]
    x = x + d; print(k, x.round(6), "mu", round(lam, 5), "h", f"{x@x - 1:.2e}")

0 [1. 1.] mu 0.0 h 1.00e+00
1 [0.75 0.75] mu 0.25 h 1.25e-01
2 [0.708333 0.708333] mu 0.40278 h 3.47e-03
3 [0.707108 0.707108] mu 0.41419 h 3.00e-06
4 [0.707107 0.707107] mu 0.41421 h 2.26e-12
5 [0.707107 0.707107] mu 0.41421 h 2.22e-16


## Điểm trong primal–dual (dạng G)

min x1²+x2² với x1+x2 ≥ 1, σ = 0.1: độ bù λ·s giảm ~10 lần mỗi bước.

In [3]:
import numpy as np
a = np.array([1., 1.]); x = np.array([1., 1.]); lam = 1.0; s = 1.0
for k in range(7):
    tau = .1*lam*s; r = np.array([*(2*x - lam*a), a@x - 1 - s, lam*s - tau])
    K = np.zeros((4, 4)); K[:2, :2] = 2*np.eye(2); K[:2, 2] = -a; K[2, :2] = a; K[2, 3] = -1; K[3, 2] = s; K[3, 3] = lam
    z = np.linalg.solve(K, -r); dx, dl, ds = z[:2], z[2], z[3]
    al = min([1.0] + [.995*(-v/dv) for v, dv in ((lam, dl), (s, ds)) if dv < 0])
    x, lam, s = x + al*dx, lam + al*dl, s + al*ds; print(k, x.round(5), round(lam, 5), f"{lam*s:.2e}")

0 [0.525 0.525] 1.05 5.25e-02
1 [0.50352 0.50352] 1.00705 7.10e-03
2 [0.50037 0.50037] 1.00075 7.49e-04
3 [0.50004 0.50004] 1.00008 7.54e-05
4 [0.5 0.5] 1.00001 7.54e-06
5 [0.5 0.5] 1.0 7.54e-07
6 [0.5 0.5] 1.0 7.54e-08


## Bài tr.41 bằng SLSQP (SQP) và trust-constr (điểm trong) (dạng H)

Nghiệm (4/√5, 2/√5), f* = (√5−2)².

In [4]:
import numpy as np
from scipy.optimize import minimize, NonlinearConstraint
f = lambda v: (v[0]-2)**2 + (v[1]-1)**2
r1 = minimize(f, [1.2, 1], method="SLSQP", constraints=[{"type": "ineq", "fun": lambda v: v[0]+v[1]-2}, {"type": "ineq", "fun": lambda v: 4-v[0]**2-v[1]**2}], options={"ftol": 1e-14})
r2 = minimize(f, [1.2, 1], method="trust-constr", constraints=[NonlinearConstraint(lambda v: [v[0]+v[1], v[0]**2+v[1]**2], [2, -np.inf], [np.inf, 4])], options={"gtol": 1e-10})
print("SLSQP:", r1.x.round(5), round(r1.fun, 6)); print("trust-constr:", r2.x.round(5), round(r2.fun, 6)); print("ly thuyet:", (np.array([4, 2])/5**.5).round(5), round((5**.5-2)**2, 6))

SLSQP:

 [1.78885 0.89443] 0.055728
trust-constr: [1.78885 0.89443] 0.055728
ly thuyet: [1.78885 0.89443] 0.055728
